In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

DB_PATH = '../data/processed/freshmart.db'
print("✅ Reorder Engine — FreshMart Retail")
print("=" * 50)

✅ Reorder Engine — FreshMart Retail


In [2]:
# Load clean inventory data
df = pd.read_csv('../data/processed/freshmart_clean.csv')
df.columns = [c.strip().lower().replace(' ', '_').replace('/', '_') for c in df.columns]
df['date'] = pd.to_datetime(df['date'])

# Load reorder recommendations from Phase 3
conn = sqlite3.connect(DB_PATH)
try:
    reorder_df = pd.read_sql("SELECT * FROM reorder_recommendations", conn)
    print("✅ Reorder recommendations loaded from database")
except:
    reorder_df = pd.read_excel('../data/output/freshmart_reorder_recommendations.xlsx',
                                sheet_name='Reorder Recommendations')
    print("✅ Reorder recommendations loaded from Excel")
conn.close()

# Load ABC classification
abc_df = pd.read_csv('../data/output/abc_classification.csv')

print(f"✅ Inventory data: {df.shape[0]:,} rows")
print(f"✅ Reorder table: {reorder_df.shape[0]} categories")
print(f"✅ ABC table: {abc_df.shape[0]} products")

✅ Reorder recommendations loaded from database
✅ Inventory data: 73,100 rows
✅ Reorder table: 5 categories
✅ ABC table: 20 products


In [3]:
# ── Configurable business parameters ──────────────────────────
LEAD_TIME_DAYS          = 7      # supplier lead time in days
SAFETY_STOCK_DAYS       = 5      # extra buffer days
MIN_ORDER_DAYS_COVER    = 30     # each order should cover at least 30 days
CRITICAL_STOCK_DAYS     = 3      # below this = CRITICAL alert
WARNING_STOCK_DAYS      = 7      # below this = WARNING alert

print("FRESHMART REORDER ENGINE — BUSINESS RULES")
print("=" * 50)
print(f"  Supplier Lead Time     : {LEAD_TIME_DAYS} days")
print(f"  Safety Stock Buffer    : {SAFETY_STOCK_DAYS} days")
print(f"  Min Order Coverage     : {MIN_ORDER_DAYS_COVER} days")
print(f"  Critical Alert Threshold: < {CRITICAL_STOCK_DAYS} days of stock")
print(f"  Warning Alert Threshold : < {WARNING_STOCK_DAYS} days of stock")

FRESHMART REORDER ENGINE — BUSINESS RULES
  Supplier Lead Time     : 7 days
  Safety Stock Buffer    : 5 days
  Min Order Coverage     : 30 days
  Critical Alert Threshold: < 3 days of stock
  Warning Alert Threshold : < 7 days of stock


In [4]:
# Calculate per-product reorder metrics
product_metrics = df.groupby(['product_id', 'category']).agg(
    avg_daily_demand  = ('units_sold', 'mean'),
    max_daily_demand  = ('units_sold', 'max'),
    std_daily_demand  = ('units_sold', 'std'),
    avg_stock         = ('inventory_level', 'mean'),
    min_stock         = ('inventory_level', 'min'),
    avg_price         = ('price', 'mean'),
    total_revenue     = ('units_sold', lambda x: (x * df.loc[x.index, 'price']).sum())
).reset_index()

# Reorder Point = (Avg Daily Demand × Lead Time) + Safety Stock
product_metrics['safety_stock']   = (product_metrics['avg_daily_demand'] * SAFETY_STOCK_DAYS).round(0)
product_metrics['reorder_point']  = (
    product_metrics['avg_daily_demand'] * LEAD_TIME_DAYS +
    product_metrics['safety_stock']
).round(0)

# Days of stock remaining
product_metrics['days_of_stock'] = (
    product_metrics['avg_stock'] / product_metrics['avg_daily_demand']
).round(1)

# Alert level
def alert_level(days):
    if days < CRITICAL_STOCK_DAYS:  return '🔴 CRITICAL'
    elif days < WARNING_STOCK_DAYS: return '🟡 WARNING'
    elif days < LEAD_TIME_DAYS:     return '🟠 REORDER NOW'
    else:                           return '🟢 OK'

product_metrics['alert'] = product_metrics['days_of_stock'].apply(alert_level)

# EOQ
ordering_cost = 500
holding_cost  = 50
product_metrics['eoq'] = np.sqrt(
    2 * product_metrics['avg_daily_demand'] * 365 * ordering_cost / holding_cost
).round(0)

# Merge ABC class
product_metrics = product_metrics.merge(
    abc_df[['product_id', 'abc_class']], on='product_id', how='left'
)

print("PRODUCT-LEVEL REORDER ANALYSIS")
print("=" * 80)
display_cols = ['product_id', 'category', 'abc_class', 'avg_daily_demand',
                'avg_stock', 'days_of_stock', 'reorder_point', 'eoq', 'alert']
print(product_metrics[display_cols].to_string(index=False))

PRODUCT-LEVEL REORDER ANALYSIS
product_id    category abc_class  avg_daily_demand  avg_stock  days_of_stock  reorder_point     eoq      alert
     P0001    Clothing         A            133.36     266.20           2.00        1601.00  987.00 🔴 CRITICAL
     P0001 Electronics         A            133.71     270.51           2.00        1605.00  988.00 🔴 CRITICAL
     P0001   Furniture         A            136.74     275.16           2.00        1641.00  999.00 🔴 CRITICAL
     P0001   Groceries         A            139.19     275.41           2.00        1670.00 1008.00 🔴 CRITICAL
     P0001        Toys         A            138.18     277.66           2.00        1658.00 1004.00 🔴 CRITICAL
     P0002    Clothing         C            135.75     268.72           2.00        1629.00  995.00 🔴 CRITICAL
     P0002 Electronics         C            130.77     267.85           2.00        1569.00  977.00 🔴 CRITICAL
     P0002   Furniture         C            131.46     280.02           2.10     

In [5]:
# Sort by urgency: Critical first, then by ABC class
priority_order = {'🔴 CRITICAL': 0, '🟠 REORDER NOW': 1, '🟡 WARNING': 2, '🟢 OK': 3}
abc_order      = {'A': 0, 'B': 1, 'C': 2}

product_metrics['alert_rank'] = product_metrics['alert'].map(priority_order)
product_metrics['abc_rank']   = product_metrics['abc_class'].map(abc_order)
priority_df = product_metrics.sort_values(['alert_rank', 'abc_rank']).reset_index(drop=True)

print("PRIORITY ACTION LIST — ORDER TO ACT")
print("=" * 80)
print("(Sorted by urgency first, then by revenue importance)\n")

for _, row in priority_df.iterrows():
    order_qty = max(row['eoq'], row['avg_daily_demand'] * MIN_ORDER_DAYS_COVER)
    print(f"{row['alert']}  [{row['abc_class']}]  {row['product_id']} ({row['category']})")
    print(f"         Days of stock: {row['days_of_stock']:.1f}d  |  "
          f"Reorder point: {row['reorder_point']:.0f} units  |  "
          f"Order qty: {order_qty:.0f} units")
    print()

PRIORITY ACTION LIST — ORDER TO ACT
(Sorted by urgency first, then by revenue importance)

🔴 CRITICAL  [A]  P0001 (Clothing)
         Days of stock: 2.0d  |  Reorder point: 1601 units  |  Order qty: 4001 units

🔴 CRITICAL  [A]  P0001 (Electronics)
         Days of stock: 2.0d  |  Reorder point: 1605 units  |  Order qty: 4011 units

🔴 CRITICAL  [A]  P0001 (Furniture)
         Days of stock: 2.0d  |  Reorder point: 1641 units  |  Order qty: 4102 units

🔴 CRITICAL  [A]  P0001 (Groceries)
         Days of stock: 2.0d  |  Reorder point: 1670 units  |  Order qty: 4176 units

🔴 CRITICAL  [A]  P0001 (Toys)
         Days of stock: 2.0d  |  Reorder point: 1658 units  |  Order qty: 4145 units

🔴 CRITICAL  [A]  P0004 (Clothing)
         Days of stock: 2.0d  |  Reorder point: 1629 units  |  Order qty: 4071 units

🔴 CRITICAL  [A]  P0004 (Electronics)
         Days of stock: 2.0d  |  Reorder point: 1584 units  |  Order qty: 3959 units

🔴 CRITICAL  [A]  P0004 (Furniture)
         Days of stock: 1.9d  

In [6]:
# Estimate revenue at risk if stockout occurs
product_metrics['daily_revenue']       = product_metrics['avg_daily_demand'] * product_metrics['avg_price']
product_metrics['revenue_at_risk_30d'] = product_metrics['daily_revenue'] * 30

critical_products = product_metrics[product_metrics['alert'].isin(['🔴 CRITICAL', '🟠 REORDER NOW'])]
warning_products  = product_metrics[product_metrics['alert'] == '🟡 WARNING']

total_risk = product_metrics['revenue_at_risk_30d'].sum()
critical_risk = critical_products['revenue_at_risk_30d'].sum() if len(critical_products) > 0 else 0

print("FINANCIAL IMPACT ANALYSIS")
print("=" * 60)
print(f"\nTotal monthly revenue at risk (all products) : ₹{total_risk:>12,.0f}")
print(f"Revenue at risk — Critical/Reorder products  : ₹{critical_risk:>12,.0f}")
print(f"Revenue at risk — Warning products           : ₹{warning_products['revenue_at_risk_30d'].sum():>12,.0f}")

print(f"\nClass A product revenue at risk:")
class_a = product_metrics[product_metrics['abc_class'] == 'A']
print(f"  ₹{class_a['revenue_at_risk_30d'].sum():,.0f} / month")

print(f"\nTop 5 highest revenue-at-risk products:")
top5 = product_metrics.nlargest(5, 'revenue_at_risk_30d')[
    ['product_id', 'category', 'abc_class', 'days_of_stock', 'daily_revenue', 'revenue_at_risk_30d', 'alert']
]
print(top5.to_string(index=False))

FINANCIAL IMPACT ANALYSIS

Total monthly revenue at risk (all products) : ₹  22,567,381
Revenue at risk — Critical/Reorder products  : ₹  22,567,381
Revenue at risk — Warning products           : ₹           0

Class A product revenue at risk:
  ₹17,008,109 / month

Top 5 highest revenue-at-risk products:
product_id    category abc_class  days_of_stock  daily_revenue  revenue_at_risk_30d      alert
     P0011    Clothing         A           1.90        8156.52            244695.64 🔴 CRITICAL
     P0012   Furniture         C           2.00        8102.46            243073.84 🔴 CRITICAL
     P0020 Electronics         A           1.90        8076.40            242292.11 🔴 CRITICAL
     P0014        Toys         A           1.90        8069.15            242074.46 🔴 CRITICAL
     P0015   Furniture         A           2.00        8046.39            241391.65 🔴 CRITICAL


In [9]:
# Ensure columns exist (in case Cell 6 was skipped)
product_metrics['daily_revenue']       = product_metrics['avg_daily_demand'] * product_metrics['avg_price']
product_metrics['revenue_at_risk_30d'] = product_metrics['daily_revenue'] * 30

excel_out = '../data/output/freshmart_master_reorder_report.xlsx'

with pd.ExcelWriter(excel_out, engine='xlsxwriter') as writer:
    wb = writer.book

    # Sheet 1: Priority Action List
    priority_out = priority_df.copy()
    # Add revenue columns if missing
    if 'daily_revenue' not in priority_out.columns:
        priority_out['daily_revenue']       = priority_out['avg_daily_demand'] * priority_out['avg_price']
        priority_out['revenue_at_risk_30d'] = priority_out['daily_revenue'] * 30

    export_cols = ['product_id','category','abc_class','avg_daily_demand',
                   'avg_stock','days_of_stock','reorder_point','eoq','alert',
                   'daily_revenue','revenue_at_risk_30d']
    
    # Only keep columns that exist
    export_cols = [c for c in export_cols if c in priority_out.columns]
    priority_out[export_cols].to_excel(writer, sheet_name='Priority Action List', index=False)

    ws = writer.sheets['Priority Action List']
    ws.set_column('A:A', 12)
    ws.set_column('B:B', 14)
    ws.set_column('C:C', 10)
    ws.set_column('D:K', 18)

    # Sheet 2: Category Summary
    reorder_df.to_excel(writer, sheet_name='Category Reorder Summary', index=False)

    # Sheet 3: ABC Classification
    abc_df.to_excel(writer, sheet_name='ABC Classification', index=False)

    # Sheet 4: Financial Impact
    fin_cols = ['product_id','category','abc_class','daily_revenue','revenue_at_risk_30d','alert']
    fin_cols = [c for c in fin_cols if c in product_metrics.columns]
    product_metrics[fin_cols].sort_values(
        'revenue_at_risk_30d', ascending=False
    ).to_excel(writer, sheet_name='Financial Impact', index=False)

print(f"✅ Master report exported → {excel_out}")

# Save to DB
conn = sqlite3.connect(DB_PATH)
product_metrics.to_sql('product_reorder_engine', conn, if_exists='replace', index=False)
priority_df.to_sql('priority_action_list', conn, if_exists='replace', index=False)
conn.close()

print("✅ Tables saved to database")
print("\n" + "=" * 60)
print("✅ 04_REORDER ENGINE COMPLETE!")
print("=" * 60)
print(f"\nOutputs in data/output/:")
for f in sorted(os.listdir('../data/output/')):
    if f != '.gitkeep':
        size = os.path.getsize(f'../data/output/{f}')
        print(f"  → {f}  ({size/1024:.1f} KB)")
print("\n✅ Ready ")

✅ Master report exported → ../data/output/freshmart_master_reorder_report.xlsx
✅ Tables saved to database

✅ 04_REORDER ENGINE COMPLETE!

Outputs in data/output/:
  → abc_classification.csv  (1.5 KB)
  → chart1_stockout_risk.png  (99.1 KB)
  → chart2_inventory_health.png  (75.4 KB)
  → chart3_revenue_trend.png  (171.0 KB)
  → chart4_inventory_heatmap.png  (156.4 KB)
  → chart5_abc_classification.png  (101.8 KB)
  → chart6_demand_forecasts.png  (1088.8 KB)
  → chart7_seasonality.png  (208.7 KB)
  → chart8_reorder_dashboard.png  (96.1 KB)
  → freshmart_master_reorder_report.xlsx  (21.3 KB)
  → freshmart_reorder_recommendations.xlsx  (31.2 KB)

✅ Ready 
